# 🧠 Notebook 05: Symbols, Identity, and Serialization

## 1. Purpose + Scope

This notebook covers the system-wide mechanisms for identity and reference:

*   **Intern Table Mechanics**: How repeated strings become unique integer IDs.
*   **Content-Addressed Identity**: Objects identified by their hash.
*   **Symbol Determinism Risks**: The danger of execution-order-dependent IDs.
*   **Canonical Export Forms**: Ensuring consistent serialized data.

## 2. Spec References

*   `spec/t81-data-types.md`
*   `include/t81/core/T81Symbol.hpp`
*   `include/t81/hash/canonhash81.hpp`

## 3. Determinism Tier

**Tier A (Strict Determinism)**: Symbol IDs are ephemeral within a process. Serialized forms must resolve symbols back to their string content or a canonical hash to maintain cross-process determinism.

## 4. Reproducibility Setup

Ensure `t81_python` is built and available in `PYTHONPATH`.

In [ ]:
import sys
import os

build_dir = os.path.abspath(os.path.join(os.getcwd(), "../build"))
if build_dir not in sys.path:
    sys.path.append(build_dir)

try:
    import t81_python
    print("✅ t81_python loaded.")
except ImportError:
    print("❌ Failed to load t81_python.")
    sys.exit(1)

## 5. Exploratory Code: Symbol Interning

Symbols convert strings to integer IDs for fast comparison. However, the specific ID depends on insertion order.

In [ ]:
# Conceptual simulation of Symbol Table behavior
class SymbolTable:
    def __init__(self):
        self._str_to_id = {}
        self._id_to_str = {}
        self._next_id = 0

    def intern(self, s):
        if s not in self._str_to_id:
            self._str_to_id[s] = self._next_id
            self._id_to_str[self._next_id] = s
            self._next_id += 1
        return self._str_to_id[s]

    def resolve(self, i):
        return self._id_to_str.get(i)

sym_table = SymbolTable()
id1 = sym_table.intern("foo")
id2 = sym_table.intern("bar")
id3 = sym_table.intern("foo")

print(f"'foo' -> {id1}")
print(f"'bar' -> {id2}")
print(f"'foo' -> {id3}")
assert id1 == id3

## 6. Serialization Hazards

If we serialize the integer IDs directly, another process might map 'bar' to ID 0 instead of 'foo'.


In [ ]:
sym_table2 = SymbolTable()
id_bar_new = sym_table2.intern("bar") # First intern in new process
print(f"Process 2: 'bar' -> {id_bar_new}")

if id_bar_new != id2:
    print("⚠️ ID Mismatch! This is why we serialize strings, not IDs.")

## 7. Canonical Export Forms

T81 uses canonical forms (e.g. sorted keys in maps) to ensure the hash of serialized data is deterministic.

In [ ]:
# Simulating canonical map serialization
data = {"z": 1, "a": 2, "m": 3}
serialized = "{"+ ",".join(f"{k}:{v}" for k, v in sorted(data.items())) + "}"
print(f"Canonical: {serialized}")

import hashlib
h = hashlib.sha256(serialized.encode()).hexdigest()
print(f"Content Hash: {h}")

## 8. Failure Mode Demonstration

Resolving an unknown ID.

In [ ]:
unknown = sym_table.resolve(999)
if unknown is None:
    print("Resolution failed for unknown ID (Expected behavior).")

## 9. Architectural Commentary

The symbol table is a process-local cache. For distributed identity, T81 uses content addressing (hashes). The bridge between local performance (integer IDs) and global identity (hashes/strings) is critical for the VM's operation.